# The Agent Loop and Its Observed Behaviour

**Technical report 2 of the Switchgear Quote Agent project**

| Item | Detail |
|---|---|
| Status | In progress |
| Covers | `run_agent` in `src/agent.py` |
| Builds on | Report 1: tool layer and safe execution |
| Data | Synthetic only |

## 1. Purpose

Report 1 described the tools and the runner that executes them. This report describes the loop that connects them to a language model: it asks the model, runs the tools the model requests, returns the results, and repeats until the model answers in text or a round limit is reached.

## 2. The loop

```
messages = [question]
repeat up to max_rounds times:
    ask the model with the full history and the tool list
    save the model's reply in the history
    if the reply has no tool request: return its text
    for each tool request:
        run it with run_tool
        save the result in the history, with the role "tool"
stop with a message if the limit is reached
```

| Element | Purpose |
|---|---|
| Full history on every call | The model keeps no memory between calls, so every call carries the whole conversation |
| Reply saved before the stop check | A tool request is stored as well as a text answer, so a result always follows its request |
| `run_tool` for every request | An unknown tool or wrong inputs return a message, not an exception |
| `max_rounds` | A run cannot continue indefinitely |

## 3. Implementation

```python
def run_agent(question: str, max_rounds: int = 5) -> str:
    """Ask the model, run the tools it requests, and stop when it answers in text."""
    messages = [{"role": "user", "content": question}]

    for round_number in range(1, max_rounds + 1):
        print(f"Round {round_number}")

        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
            think=False,
        )
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content

        for call in response.message.tool_calls:
            name = call.function.name
            result = run_tool(name, call.function.arguments)
            print(f"  {name}({call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "tool_name": name, "content": result})
    return "Stopped: reached the round limit."
```

Thinking mode is switched off (`think=False`). In an earlier test, a model with a small token budget spent all of it on reasoning and returned an empty answer.

## 4. Environment

| Item | Detail |
|---|---|
| Model | Gemma 4, small size, served by Ollama |
| Hardware | Google Colab, Tesla T4 |
| Files | `tools.py` and `agent.py`, copied into the session |

## 5. Observed runs

| Question | Output | Reading |
|---|---|---|
| What is the capital of France? | `Round 1`, then a text answer | No tool needed. The stop rule ended the run in one round. |
| What is the price of one busbar? | `Round 1`, `lookup_price({'part_name': 'busbar'}) -> 95.0`, `Round 2`, then a text answer of 95.0 | One tool request, then an answer based on its result |
| How much do 3 busbars cost in total? | `Round 1`, `lookup_price(...) -> 95.0`, `Round 2`, `line_total({'quantity': 3, 'unit_price': 95}) -> 285`, `Round 3`, then a text answer of 285.00 | Two tools in sequence. The first result became the second tool's input. |

**Direct checks of `run_tool`**

| Request | Result |
|---|---|
| Unknown tool `make_coffee` | Message listing the three valid tools |
| `lookup_price` for `fuse` | Message listing the five valid parts |
| `line_total` without `quantity` | Message containing Python's error text |

**Check applied to every answer:** each number in the final text must match a logged tool result. `95.0` and `285` do.

## 6. Findings

1. **A stale module copy hid the loop.** An earlier run showed two rounds and no tool lines. The cause was an outdated `agent.py` loaded in the session, not the model. Writing the new file and reloading the module resolved it. Checks that distinguish the cases: a text search of the file, and `inspect.getsource` on the loaded function.
2. **Tool descriptions changed behaviour.** In an earlier prototype, a short description without the valid part names made the model ask a clarifying question. Listing the names in the docstring led to a tool call.
3. **Formatting is chosen by the model.** The tool returned `285`. The final text said `285.00`. Presentation of amounts should be done by code before it reaches the user.
4. **Types are not guaranteed.** `lookup_price` returns text. The model passed `95` as a number to `line_total`. This worked in the observed run but is not enforced.

## 7. Limitations

| Limitation | Note |
|---|---|
| Few runs | Three questions on one model. This shows the loop works, not how often. |
| Small model | It may skip a tool and answer from its own knowledge. The log check catches this. |
| No schema on the final answer | Output is free text. A typed result is planned. |
| One price per part type | Real quotes depend on ratings and sizes |
| Synthetic data | Results say nothing about real pricing |

## 8. Next steps

| Step | Content |
|---|---|
| 1 | A typed result for the final answer, with validation |
| 2 | Retrieval over a small synthetic catalogue, with the source shown |
| 3 | Reading drawing images |